In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = r'agg_custom.csv'

df = pd.read_csv(file_path, encoding='utf-8')

df

,大区,城市,商品名称,单价,数量,总价,品类,是否包邮
0,华北,北京,手机,2999,2,5998,数码,True
1,华北,北京,键盘,899,3,2697,数码,True
2,华东,上海,耳机,399,5,1995,数码,False
3,华南,广州,卫衣,199,10,1990,服饰,True
4,华南,深圳,水杯,59,20,1180,日用品,False
5,华东,上海,鼠标,99,8,792,数码,False
6,华南,广州,裤子,299,6,1794,服饰,True
7,华南,深圳,毛巾,19,15,285,日用品,False
8,华北,北京,平板,1999,4,7996,数码,True
9,华东,上海,音箱,299,7,2093,数码,False


In [3]:
group_cat = df.groupby('品类')
group_cat

In [4]:
def range_diff(x):
    return x.max() - x.min()

agg_range = group_cat['总价'].agg(range_diff)
agg_range

品类
数码     7204
日用品     895
服饰      196
Name: 总价, dtype: int64

In [5]:
def rate_200(x):
    return (x > 200).sum() / len(x) * 100

agg_rate = group_cat['单价'].agg(rate_200).round(2)
agg_rate

品类
数码     83.33
日用品     0.00
服饰     50.00
Name: 单价, dtype: float64

In [6]:
def mean_std(x):
    return f"{x.mean():.2f}±{x.std():.2f}"

def free_rate(x):
    return x.sum() / len(x) * 100

In [8]:
agg_multi_custom = group_cat.agg(
    总价极值差 = ('总价', range_diff),
    单价达标率 = ('单价', rate_200),
    数量均值标准差 = ('数量', mean_std),
    包邮率 = ('是否包邮', free_rate)
).round(2)

agg_multi_custom

,总价极值差,单价达标率,数量均值标准差,包邮率
品类,,,,
数码,7204,83.33,4.83±2.32,50.0
日用品,895,0.00,17.50±3.54,0.0
服饰,196,50.00,8.00±2.83,100.0


In [9]:
agg_lambda = group_cat['总价'].agg(
    中位数 = lambda x: np.median(x),
    均值_2倍 = lambda x: x.mean() * 2,
    占比 = lambda x: x.sum() / df['总价'].sum() * 100
)

agg_lambda

,中位数,均值_2倍,占比
品类,,,
数码,2395.0,7190.333333,80.428784
日用品,732.5,1465.000000,5.462342
服饰,1892.0,3784.000000,14.108874


In [10]:
df

,大区,城市,商品名称,单价,数量,总价,品类,是否包邮
0,华北,北京,手机,2999,2,5998,数码,True
1,华北,北京,键盘,899,3,2697,数码,True
2,华东,上海,耳机,399,5,1995,数码,False
3,华南,广州,卫衣,199,10,1990,服饰,True
4,华南,深圳,水杯,59,20,1180,日用品,False
5,华东,上海,鼠标,99,8,792,数码,False
6,华南,广州,裤子,299,6,1794,服饰,True
7,华南,深圳,毛巾,19,15,285,日用品,False
8,华北,北京,平板,1999,4,7996,数码,True
9,华东,上海,音箱,299,7,2093,数码,False


In [11]:
def cv(x):
    return x.std() / x.mean() if x.mean() != 0 else 0

group_multi = df.groupby(['大区', '品类'])
agg_multi_group = group_multi['总价'].agg(
    变异系数 = cv,
    总价合计 = 'sum'
).round(3)

agg_multi_group

变异系数   总价合计
大区 品类               
华东 数码   0.445   4880
华北 数码   0.481  16691
华南 日用品  0.864   1465
   服饰   0.073   3784

In [12]:
agg_filter = agg_multi_custom[agg_multi_custom['总价极值差'] > 1000]
agg_filter

,总价极值差,单价达标率,数量均值标准差,包邮率
品类,,,,
数码,7204,83.33,4.83±2.32,50.0
